In [5]:
%load_ext autoreload
%autoreload 2

import email_utils

# Pass custom Excel path or batch size if necessary
email_utils.send_bulk_emails_safely(
    batch_size=400
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- Starting Bulk Email Draft Creation Process ---
Target Excel Path: F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\Email list.xlsx
Target Attachment Path: F:\Chimney Work\Marketing\Brochure\Nirmanshila Construction - Brochure.pdf
Reading Excel file...
Total rows found in Excel: 400
Attachment 'Nirmanshila Construction - Brochure.pdf' loaded successfully.
Filtering eligible emails based on criteria (No DND, sent >= 7 days ago)...
Filtering complete:
 - Skipped (DND): 0
 - Skipped (Sent within last 7 days): 0
 - Total Eligible Emails: 400
Connecting to Gmail IMAP server (imap.gmail.com)...
Successfully logged into Gmail via IMAP.

Processing single draft (400 recipients)...
Saving single draft to '[Gmail]/Drafts'...
Draft saved successfully. Updating Excel statuses to 'Sent' and current date...
Excel file updated.


c:\Users\Amit\projects\NC Project\2- NC Email Marketing\email_utils.py:212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-08-10' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "Last Sent Date"] = timestamp_str



--- Process Finished ---
Draft saved to Drafts successfully!


In [6]:
%load_ext autoreload
%autoreload 2
from Bounceback_utils import update_excel_with_bounces

update_excel_with_bounces()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[INFO] Starting email processing...
[INFO] Reading Excel file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\EmailMasterDB.xlsx
[INFO] Loaded 3775 emails from Excel.
[INFO] Connecting to Gmail IMAP (imap.gmail.com:993) for contact.nirmanshila@gmail.com...
[INFO] Connected successfully. Searching INBOX...
[INFO] Found 2 messages in INBOX. Parsing messages...
[INFO] Processing email deletions...
[INFO] Permanently expunging 2 message(s) from INBOX...
[SUCCESS] Deleted 2 bounce/DND email(s).
[INFO] IMAP session closed successfully.
[INFO] Updating Excel dataset...
[SUCCESS] Excel file updated successfully with 101 DND statuses and 0 alternate emails.
[SUCCESS] Execution completed.


In [ ]:
# update the masterDB email for last sent and status
import os
import pandas as pd

# Fetch file paths
mdb_path = os.environ.get("emailMDB")
out_path = os.environ.get("email")

if not mdb_path or not out_path:
    raise ValueError(
        "Environment variables 'emailMDB' and 'email' must be set."
    )

# Load both files
df_mdb = pd.read_excel(mdb_path)
df_sent = pd.read_excel(out_path)

# Columns to sync back to the master database
COLS_TO_UPDATE = ["Last Sent Date"]

# Replace 'Email ID' with your actual unique identifier column name if different
UNIQUE_KEY = "Email ID"

if UNIQUE_KEY in df_mdb.columns and UNIQUE_KEY in df_sent.columns:
    # Set index to unique key to merge updates directly
    df_mdb.set_index(UNIQUE_KEY, inplace=True)
    df_sent.set_index(UNIQUE_KEY, inplace=True)

    # Update both 'Last Sent Date' and 'Email ID Status' in the main dataset
    df_mdb.update(df_sent[COLS_TO_UPDATE])

    # Reset index back to normal layout
    df_mdb.reset_index(inplace=True)
else:
    # Fallback to positional index if no specific unique key column exists
    df_mdb.update(df_sent[COLS_TO_UPDATE])

# Save the updated main database back to emailMDB
df_mdb.to_excel(mdb_path, index=False)
print(
    f"Successfully synced updated 'Last Sent Date' back to: {mdb_path}"
)

# Clean up temporary daily batch file
os.remove(out_path)

Successfully synced updated 'Last Sent Date' and 'Email ID Status' back to: F:\Chimney Work\Marketing\LeadGen\Data Architecture\3 - Gold\Email list\EmailMasterDB.xlsx
